<a href="https://colab.research.google.com/github/SoyebAhmed5/flyrank-internship-ml/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SoyebAhmed5/flyrank-internship-ml/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [21]:

!pip -q install duckdb

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

In [31]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


import duckdb

from google.colab import userdata



hf_token = userdata.get("HF_TOKEN")

con = duckdb.connect()
con.execute("INSTALL httpfs")
con.execute("LOAD httpfs")

# Safely escape the token for SQL; never print it.
safe_token = hf_token.replace("'", "''")
con.execute(
    f"CREATE SECRET (TYPE huggingface, TOKEN '{safe_token}')"
)

rel = "hf://datasets/FlyRank/internship-warehouse"
table_path = (
    f"{rel}/fact_content_daily_performance/**/*.parquet"
)

print("DuckDB connected and secret configured.")


DuckDB connected and secret configured.


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

In [32]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


schema = con.execute(
    f"""
    DESCRIBE
    SELECT *
    FROM read_parquet('{table_path}')
    LIMIT 1
    """
).df()

display(schema)



,column_name,column_type,null,key,default,extra
0,report_date,DATE,YES,None,None,None
1,client_hash_id,VARCHAR,YES,None,None,None
2,content_hash_id,VARCHAR,YES,None,None,None
3,client_has_gsc,BOOLEAN,YES,None,None,None
4,client_has_ga4,BOOLEAN,YES,None,None,None
5,gsc_data_available,BOOLEAN,YES,None,None,None
6,ga4_data_available,BOOLEAN,YES,None,None,None
7,gsc_impressions,BIGINT,YES,None,None,None
8,gsc_clicks,BIGINT,YES,None,None,None
9,gsc_sum_position,BIGINT,YES,None,None,None


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [33]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

schema = con.execute("""
    DESCRIBE
    SELECT *
    FROM read_parquet(
        'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet'
    )
    LIMIT 1
""").df()

display(schema)

,column_name,column_type,null,key,default,extra
0,report_date,DATE,YES,None,None,None
1,client_hash_id,VARCHAR,YES,None,None,None
2,content_hash_id,VARCHAR,YES,None,None,None
3,client_has_gsc,BOOLEAN,YES,None,None,None
4,client_has_ga4,BOOLEAN,YES,None,None,None
5,gsc_data_available,BOOLEAN,YES,None,None,None
6,ga4_data_available,BOOLEAN,YES,None,None,None
7,gsc_impressions,BIGINT,YES,None,None,None
8,gsc_clicks,BIGINT,YES,None,None,None
9,gsc_sum_position,BIGINT,YES,None,None,None


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

### Data limitations

1. **Unbalanced history:** Each client has a different length of historical data. Therefore, I cannot assume that every client has the same observation period or compare their histories without considering this difference.

2. **GSC-only early rows:** Some early observations may contain Google Search Console (GSC) data without Google Analytics 4 (GA4) data. I cannot treat missing GA4 measurements as zero or make complete cross-source comparisons for those rows.

3. **Overlapping time windows:** The 90-day query-level window includes overlapping last-30-day and previous-30-day periods. These are not independent observations, so I must avoid double-counting them or treating overlapping measurements as separate evidence.

4. **Causality limitation:** Historical observations can show associations and directional changes, but they cannot establish that refreshing a piece of content caused its performance to improve or decline.

5. **Generalization limitation:** Results from the March 2026 development data may not generalize to other months, clients, or future conditions. The June 2026 test month must remain separate from development.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.